In [ ]:
# Author(s): TODO
# Project: ode_regime
# Date: May 2026

# ODE System, code `ode_regime`

This notebook implements formulas (1)--(3) from the subject, then compares the requested variants:

- baseline ODE with `g Y_j`,
- deterministic square-root growth with `g sqrt(Y_j)`,
- stochastic square-root model with independent Brownian increments.

The explicit time stepping loop is necessary because the model is time-dependent through the classes and the stochastic variant. The class updates are vectorized over `j`.

In [ ]:
import numpy as np

In [ ]:
from pathlib import Path
from html import escape

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    plt = None
    HAS_MATPLOTLIB = False

PLOT_DIR = Path("generated_figures")
PLOT_DIR.mkdir(exist_ok=True)
SVG_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]


def emit_svg(svg_text, filename):
    """Display SVG in a notebook when possible; always save it as a fallback."""
    path = PLOT_DIR / filename
    path.write_text(svg_text, encoding="utf-8")
    try:
        from IPython.display import SVG, display
        display(SVG(svg_text))
    except Exception:
        print(f"wrote {path}")


def scale_values(values, in_min, in_max, out_min, out_max):
    values = np.asarray(values, dtype=float)
    if np.isclose(in_min, in_max):
        return np.full_like(values, 0.5 * (out_min + out_max), dtype=float)
    return out_min + (values - in_min) * (out_max - out_min) / (in_max - in_min)


def render_line_svg(x, series, labels, title, x_label, y_label, filename, log_y=False):
    width, height = 900, 520
    left, right, top, bottom = 80, 30, 55, 75
    plot_left, plot_right = left, width - right
    plot_top, plot_bottom = top, height - bottom
    x = np.asarray(x, dtype=float)
    y_series = []
    for values in series:
        y = np.asarray(values, dtype=float)
        if log_y:
            y = np.log10(np.maximum(y, 1.0e-300))
        y_series.append(y)
    all_y = np.concatenate(y_series)
    x_min, x_max = float(np.min(x)), float(np.max(x))
    y_min, y_max = float(np.min(all_y)), float(np.max(all_y))
    y_padding = 0.05 * max(y_max - y_min, 1.0)
    y_min -= y_padding
    y_max += y_padding

    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
    parts.append('<rect width="100%" height="100%" fill="white"/>')
    parts.append(f'<text x="{width/2}" y="28" text-anchor="middle" font-size="20" font-family="sans-serif">{escape(title)}</text>')
    parts.append(f'<line x1="{plot_left}" y1="{plot_bottom}" x2="{plot_right}" y2="{plot_bottom}" stroke="black"/>')
    parts.append(f'<line x1="{plot_left}" y1="{plot_top}" x2="{plot_left}" y2="{plot_bottom}" stroke="black"/>')
    parts.append(f'<text x="{width/2}" y="{height-25}" text-anchor="middle" font-size="14" font-family="sans-serif">{escape(x_label)}</text>')
    y_axis_label = f'log10({y_label})' if log_y else y_label
    parts.append(f'<text x="18" y="{height/2}" transform="rotate(-90 18 {height/2})" text-anchor="middle" font-size="14" font-family="sans-serif">{escape(y_axis_label)}</text>')
    for tick_value in np.linspace(x_min, x_max, 5):
        tick_x = scale_values(tick_value, x_min, x_max, plot_left, plot_right)
        parts.append(f'<line x1="{tick_x:.1f}" y1="{plot_bottom}" x2="{tick_x:.1f}" y2="{plot_bottom+5}" stroke="black"/>')
        parts.append(f'<text x="{tick_x:.1f}" y="{plot_bottom+22}" text-anchor="middle" font-size="11" font-family="sans-serif">{tick_value:.2g}</text>')
    for tick_value in np.linspace(y_min, y_max, 5):
        tick_y = scale_values(tick_value, y_min, y_max, plot_bottom, plot_top)
        parts.append(f'<line x1="{plot_left-5}" y1="{tick_y:.1f}" x2="{plot_left}" y2="{tick_y:.1f}" stroke="black"/>')
        parts.append(f'<text x="{plot_left-9}" y="{tick_y+4:.1f}" text-anchor="end" font-size="11" font-family="sans-serif">{tick_value:.2g}</text>')
    x_pixels = scale_values(x, x_min, x_max, plot_left, plot_right)
    for index, (y, label) in enumerate(zip(y_series, labels)):
        y_pixels = scale_values(y, y_min, y_max, plot_bottom, plot_top)
        points = " ".join(f"{xp:.1f},{yp:.1f}" for xp, yp in zip(x_pixels, y_pixels))
        color = SVG_COLORS[index % len(SVG_COLORS)]
        parts.append(f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2"/>')
        legend_y = top + 22 * index
        parts.append(f'<line x1="{plot_right-170}" y1="{legend_y}" x2="{plot_right-145}" y2="{legend_y}" stroke="{color}" stroke-width="3"/>')
        parts.append(f'<text x="{plot_right-138}" y="{legend_y+4}" font-size="12" font-family="sans-serif">{escape(label)}</text>')
    parts.append('</svg>')
    emit_svg("\n".join(parts), filename)


def render_heatmap_svg(matrix, title, x_label, y_label, filename, max_columns=220):
    matrix = np.asarray(matrix, dtype=float)
    if matrix.shape[1] > max_columns:
        column_index = np.linspace(0, matrix.shape[1] - 1, max_columns).astype(int)
        matrix = matrix[:, column_index]
    rows, columns = matrix.shape
    width, height = 900, 430
    left, right, top, bottom = 75, 25, 55, 65
    plot_width, plot_height = width - left - right, height - top - bottom
    cell_width = plot_width / columns
    cell_height = plot_height / rows
    min_value, max_value = float(np.min(matrix)), float(np.max(matrix))
    span = max(max_value - min_value, 1.0e-15)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
    parts.append('<rect width="100%" height="100%" fill="white"/>')
    parts.append(f'<text x="{width/2}" y="28" text-anchor="middle" font-size="20" font-family="sans-serif">{escape(title)}</text>')
    for row in range(rows):
        for column in range(columns):
            normalized = (matrix[row, column] - min_value) / span
            red = int(255 * (1.0 - normalized))
            green = int(255 * (1.0 - normalized))
            blue = 255
            x = left + column * cell_width
            y = top + (rows - 1 - row) * cell_height
            parts.append(f'<rect x="{x:.2f}" y="{y:.2f}" width="{cell_width+0.2:.2f}" height="{cell_height+0.2:.2f}" fill="rgb({red},{green},{blue})"/>')
    parts.append(f'<rect x="{left}" y="{top}" width="{plot_width}" height="{plot_height}" fill="none" stroke="black"/>')
    parts.append(f'<text x="{width/2}" y="{height-22}" text-anchor="middle" font-size="14" font-family="sans-serif">{escape(x_label)}</text>')
    parts.append(f'<text x="18" y="{height/2}" transform="rotate(-90 18 {height/2})" text-anchor="middle" font-size="14" font-family="sans-serif">{escape(y_label)}</text>')
    parts.append('</svg>')
    emit_svg("\n".join(parts), filename)


def render_grouped_bar_svg(categories, series, labels, title, x_label, y_label, filename):
    categories = np.asarray(categories)
    series = [np.asarray(values, dtype=float) for values in series]
    width, height = 900, 480
    left, right, top, bottom = 75, 30, 55, 75
    plot_left, plot_right = left, width - right
    plot_top, plot_bottom = top, height - bottom
    y_max = max(float(np.max(values)) for values in series)
    y_max = max(y_max, 1.0e-15)
    group_count = len(categories)
    series_count = len(series)
    group_width = (plot_right - plot_left) / group_count
    bar_width = 0.78 * group_width / series_count
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
    parts.append('<rect width="100%" height="100%" fill="white"/>')
    parts.append(f'<text x="{width/2}" y="28" text-anchor="middle" font-size="20" font-family="sans-serif">{escape(title)}</text>')
    parts.append(f'<line x1="{plot_left}" y1="{plot_bottom}" x2="{plot_right}" y2="{plot_bottom}" stroke="black"/>')
    parts.append(f'<line x1="{plot_left}" y1="{plot_top}" x2="{plot_left}" y2="{plot_bottom}" stroke="black"/>')
    for group_index, category in enumerate(categories):
        group_x = plot_left + group_index * group_width
        for series_index, values in enumerate(series):
            value = values[group_index]
            bar_height = (value / y_max) * (plot_bottom - plot_top)
            x = group_x + 0.11 * group_width + series_index * bar_width
            y = plot_bottom - bar_height
            color = SVG_COLORS[series_index % len(SVG_COLORS)]
            parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_width:.1f}" height="{bar_height:.1f}" fill="{color}" opacity="0.85"/>')
        label_x = group_x + 0.5 * group_width
        parts.append(f'<text x="{label_x:.1f}" y="{plot_bottom+20}" text-anchor="middle" font-size="11" font-family="sans-serif">{escape(str(category))}</text>')
    for tick_value in np.linspace(0.0, y_max, 5):
        tick_y = scale_values(tick_value, 0.0, y_max, plot_bottom, plot_top)
        parts.append(f'<line x1="{plot_left-5}" y1="{tick_y:.1f}" x2="{plot_left}" y2="{tick_y:.1f}" stroke="black"/>')
        parts.append(f'<text x="{plot_left-9}" y="{tick_y+4:.1f}" text-anchor="end" font-size="11" font-family="sans-serif">{tick_value:.2g}</text>')
    parts.append(f'<text x="{width/2}" y="{height-25}" text-anchor="middle" font-size="14" font-family="sans-serif">{escape(x_label)}</text>')
    parts.append(f'<text x="18" y="{height/2}" transform="rotate(-90 18 {height/2})" text-anchor="middle" font-size="14" font-family="sans-serif">{escape(y_label)}</text>')
    for index, label in enumerate(labels):
        legend_x = plot_right - 180
        legend_y = top + 22 * index
        color = SVG_COLORS[index % len(SVG_COLORS)]
        parts.append(f'<rect x="{legend_x}" y="{legend_y-10}" width="16" height="12" fill="{color}" opacity="0.85"/>')
        parts.append(f'<text x="{legend_x+23}" y="{legend_y}" font-size="12" font-family="sans-serif">{escape(label)}</text>')
    parts.append('</svg>')
    emit_svg("\n".join(parts), filename)


def render_hist_grid_svg(forward_samples, ode_samples, plot_steps, dt_value, bins, filename):
    columns = 3
    panel_width, panel_height = 330, 250
    rows = int(np.ceil(len(plot_steps) / columns))
    width, height = columns * panel_width, rows * panel_height
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
    parts.append('<rect width="100%" height="100%" fill="white"/>')
    for panel_index, step in enumerate(plot_steps):
        row, column = divmod(panel_index, columns)
        x0, y0 = column * panel_width, row * panel_height
        left, right, top, bottom = x0 + 45, x0 + panel_width - 12, y0 + 38, y0 + panel_height - 35
        ode_step = step if step in ode_samples else 0
        hist_forward, edges = np.histogram(forward_samples[step], bins=bins, density=True)
        hist_ode, _ = np.histogram(ode_samples[ode_step], bins=bins, density=True)
        y_max = max(float(hist_forward.max()), float(hist_ode.max()), 1.0e-15)
        bar_width = (right - left) / hist_forward.size
        title = "t = 0 (ODE at eps)" if step == 0 else f"t = {step * dt_value:.3f}"
        parts.append(f'<text x="{x0 + panel_width/2}" y="{y0+22}" text-anchor="middle" font-size="15" font-family="sans-serif">{escape(title)}</text>')
        parts.append(f'<line x1="{left}" y1="{bottom}" x2="{right}" y2="{bottom}" stroke="black"/>')
        parts.append(f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="black"/>')
        for index, (f_value, o_value) in enumerate(zip(hist_forward, hist_ode)):
            x = left + index * bar_width
            f_height = (f_value / y_max) * (bottom - top)
            o_height = (o_value / y_max) * (bottom - top)
            parts.append(f'<rect x="{x:.1f}" y="{bottom-f_height:.1f}" width="{bar_width:.1f}" height="{f_height:.1f}" fill="#1f77b4" opacity="0.48"/>')
            parts.append(f'<rect x="{x:.1f}" y="{bottom-o_height:.1f}" width="{bar_width:.1f}" height="{o_height:.1f}" fill="#ff7f0e" opacity="0.48"/>')
    parts.append(f'<rect x="18" y="12" width="14" height="10" fill="#1f77b4" opacity="0.48"/><text x="38" y="22" font-size="13" font-family="sans-serif">forward SDE</text>')
    parts.append(f'<rect x="140" y="12" width="14" height="10" fill="#ff7f0e" opacity="0.48"/><text x="160" y="22" font-size="13" font-family="sans-serif">backward ODE</text>')
    parts.append('</svg>')
    emit_svg("\n".join(parts), filename)

In [ ]:
# Parameters from the subject.
T = 300.0
J = 10
lambda_exponent_min = -6.0
lambda_exponent_max = 3.0
lambda_values = 10.0 ** np.linspace(lambda_exponent_min, lambda_exponent_max, J)
q_values = np.full(J, 1.0 / J)
p = 0.95
m = 10.0 ** (-8.0)
g = 0.1
X_init = 10.0 ** 6.0
X0 = np.full(J, X_init / J)
Y0 = np.zeros(J)

# The largest lambda is 1000, so explicit Euler needs dt <= 1 / max(lambda)
# to keep the loss term from making X negative. This value is derived below.
requested_dt = 1.0e-3
n_store_target = 1200
random_seed = 202605

In [ ]:
def positive_euler_step_size(T_value, requested_dt_value, lambda_array):
    """Choose a time step satisfying dt * max(lambda_j) <= 1 for positivity of X."""
    max_lambda = np.max(lambda_array)
    dt_value = min(requested_dt_value, 1.0 / max_lambda)
    n_steps = int(np.ceil(T_value / dt_value))
    dt_value = T_value / n_steps
    while dt_value * max_lambda > 1.0:
        n_steps += 1
        dt_value = T_value / n_steps
    return dt_value, n_steps


def safe_shares(values):
    """Return class shares row by row, using zero when the total is zero."""
    totals = values.sum(axis=1, keepdims=True)
    return np.divide(values, totals, out=np.zeros_like(values), where=totals > 0.0)


def simulate_regime(
    T_value,
    J_value,
    lambda_array,
    q_array,
    p_value,
    m_value,
    g_value,
    X_initial,
    Y_initial,
    requested_dt_value,
    n_store_target_value,
    y_mode="linear",
    seed=None,
):
    """Simulate formulas (1)--(3) and the requested Y variants.

    y_mode="linear" uses g Y_j.
    y_mode="sqrt" uses g sqrt(Y_j).
    y_mode="sqrt_sde" adds sqrt(g sqrt(Y_j)) dW_t^j with full truncation.
    """
    dt_value, n_steps = positive_euler_step_size(T_value, requested_dt_value, lambda_array)
    store_stride = max(1, n_steps // n_store_target_value)
    rng = np.random.default_rng(seed)

    X = X_initial.astype(float).copy()
    Y = Y_initial.astype(float).copy()
    stored_t = [0.0]
    stored_X = [X.copy()]
    stored_Y = [Y.copy()]

    for step in range(1, n_steps + 1):
        X_old = X.copy()
        Y_old = np.maximum(Y, 0.0)
        L_X = lambda_array @ X_old

        # Formula (1): the source term is non-negative and the chosen dt keeps X non-negative.
        X = X_old + dt_value * (
            -lambda_array * X_old + p_value * (1.0 - m_value) * q_array * L_X
        )
        X = np.maximum(X, 0.0)

        # Formula (2) and its requested variants.
        immigration = p_value * m_value * lambda_array * X_old
        if y_mode == "linear":
            Y = Y_old + dt_value * (immigration + g_value * Y_old)
        elif y_mode == "sqrt":
            Y = Y_old + dt_value * (immigration + g_value * np.sqrt(Y_old))
        elif y_mode == "sqrt_sde":
            diffusion = np.sqrt(np.maximum(g_value * np.sqrt(Y_old), 0.0))
            brownian_step = np.sqrt(dt_value) * rng.normal(size=J_value)
            Y = Y_old + dt_value * (immigration + g_value * np.sqrt(Y_old)) + diffusion * brownian_step
        else:
            raise ValueError("unknown y_mode")

        # Full truncation/projection enforces the non-negativity constraint from the subject.
        Y = np.maximum(Y, 0.0)

        if step % store_stride == 0 or step == n_steps:
            stored_t.append(step * dt_value)
            stored_X.append(X.copy())
            stored_Y.append(Y.copy())

    return {
        "t": np.array(stored_t),
        "X": np.array(stored_X),
        "Y": np.array(stored_Y),
        "dt": dt_value,
        "n_steps": n_steps,
        "mode": y_mode,
    }

In [ ]:
baseline = simulate_regime(
    T, J, lambda_values, q_values, p, m, g, X0, Y0,
    requested_dt, n_store_target, y_mode="linear", seed=random_seed,
)
sqrt_variant = simulate_regime(
    T, J, lambda_values, q_values, p, m, g, X0, Y0,
    requested_dt, n_store_target, y_mode="sqrt", seed=random_seed,
)
sde_variant = simulate_regime(
    T, J, lambda_values, q_values, p, m, g, X0, Y0,
    requested_dt, n_store_target, y_mode="sqrt_sde", seed=random_seed,
)

print(f"dt = {baseline['dt']:.3e}, steps = {baseline['n_steps']}")
print(f"minimum baseline X = {baseline['X'].min():.3e}")
print(f"minimum baseline Y = {baseline['Y'].min():.3e}")
print(f"minimum stochastic Y = {sde_variant['Y'].min():.3e}")

In [ ]:
if HAS_MATPLOTLIB:
    plt.figure(figsize=(8, 5))
    plt.semilogy(baseline["t"], baseline["X"].sum(axis=1), label="sum X, baseline")
    plt.semilogy(baseline["t"], baseline["Y"].sum(axis=1), label="sum Y, gY")
    plt.semilogy(sqrt_variant["t"], sqrt_variant["Y"].sum(axis=1), label="sum Y, g sqrt(Y)")
    plt.semilogy(sde_variant["t"], sde_variant["Y"].sum(axis=1), label="sum Y, stochastic", alpha=0.8)
    plt.xlabel("time")
    plt.ylabel("total population / mass")
    plt.title("Evolution of total X and Y classes")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    render_line_svg(
        baseline["t"],
        [
            baseline["X"].sum(axis=1),
            baseline["Y"].sum(axis=1),
            sqrt_variant["Y"].sum(axis=1),
            sde_variant["Y"].sum(axis=1),
        ],
        ["sum X, baseline", "sum Y, gY", "sum Y, g sqrt(Y)", "sum Y, stochastic"],
        "Evolution of total X and Y classes",
        "time",
        "total population / mass",
        "ode_totals.svg",
        log_y=True,
    )

In [ ]:
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    x_share_image = axes[0].imshow(
        safe_shares(baseline["X"]).T,
        aspect="auto",
        origin="lower",
        extent=[baseline["t"][0], baseline["t"][-1], 1, J],
    )
    axes[0].set_title("Baseline X class shares")
    axes[0].set_xlabel("time")
    axes[0].set_ylabel("class j")
    fig.colorbar(x_share_image, ax=axes[0])
    y_share_image = axes[1].imshow(
        safe_shares(baseline["Y"]).T,
        aspect="auto",
        origin="lower",
        extent=[baseline["t"][0], baseline["t"][-1], 1, J],
    )
    axes[1].set_title("Baseline Y class shares")
    axes[1].set_xlabel("time")
    axes[1].set_ylabel("class j")
    fig.colorbar(y_share_image, ax=axes[1])
    plt.show()
else:
    render_heatmap_svg(safe_shares(baseline["X"]).T, "Baseline X class shares", "time", "class j", "ode_x_shares.svg")
    render_heatmap_svg(safe_shares(baseline["Y"]).T, "Baseline Y class shares", "time", "class j", "ode_y_shares.svg")

In [ ]:
class_index = np.arange(1, J + 1)
final_X_share = safe_shares(baseline["X"])[-1]
final_Y_share = safe_shares(baseline["Y"])[-1]
final_sqrt_Y_share = safe_shares(sqrt_variant["Y"])[-1]
final_sde_Y_share = safe_shares(sde_variant["Y"])[-1]

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].bar(class_index, final_X_share)
    axes[0].set_title("Final X histogram by class")
    axes[0].set_xlabel("class j")
    axes[0].set_ylabel("share")
    bar_width = 0.25
    axes[1].bar(class_index - bar_width, final_Y_share, width=bar_width, label="gY")
    axes[1].bar(class_index, final_sqrt_Y_share, width=bar_width, label="g sqrt(Y)")
    axes[1].bar(class_index + bar_width, final_sde_Y_share, width=bar_width, label="stochastic")
    axes[1].set_title("Final Y histogram by class")
    axes[1].set_xlabel("class j")
    axes[1].set_ylabel("share")
    axes[1].legend()
    plt.show()
else:
    render_grouped_bar_svg(class_index, [final_X_share], ["X"], "Final X histogram by class", "class j", "share", "ode_final_x.svg")
    render_grouped_bar_svg(
        class_index,
        [final_Y_share, final_sqrt_Y_share, final_sde_Y_share],
        ["gY", "g sqrt(Y)", "stochastic"],
        "Final Y histogram by class",
        "class j",
        "share",
        "ode_final_y.svg",
    )

## Search for a middle-maximum regime

The subject asks whether changing the parameters can produce a unimodal `Y_j` histogram whose maximum is in the middle. The short search below is intentionally bounded so the notebook remains runnable. The criterion allows a small tolerance because several adjacent middle classes may be almost tied.

In [ ]:
def is_middle_unimodal(values, tolerance=5.0e-3):
    shares = values / values.sum() if values.sum() > 0.0 else values
    near_max = np.flatnonzero(shares >= shares.max() - tolerance)
    internal_near_max = near_max[(near_max > 0) & (near_max < shares.size - 1)]
    if internal_near_max.size > 0:
        center = 0.5 * (shares.size - 1)
        peak = int(internal_near_max[np.argmin(np.abs(internal_near_max - center))])
    else:
        peak = int(np.argmax(shares))
    increasing_left = np.all(np.diff(shares[: peak + 1]) >= -tolerance)
    decreasing_right = np.all(np.diff(shares[peak:]) <= tolerance)
    peak_is_internal = 0 < peak < shares.size - 1
    return peak_is_internal and increasing_left and decreasing_right, peak, shares

candidate_parameters = [
    {
        "T_value": 5.0,
        "p_value": 0.50,
        "g_value": 0.0,
        "lambda_min": -1.0,
        "lambda_max": 2.0,
        "label": "short horizon, lower p, no Y growth",
    },
    {
        "T_value": 5.0,
        "p_value": 0.95,
        "g_value": 0.0,
        "lambda_min": -3.0,
        "lambda_max": 2.0,
        "label": "short horizon, wide lambda range, no Y growth",
    },
    {
        "T_value": 20.0,
        "p_value": 0.75,
        "g_value": 0.01,
        "lambda_min": -2.0,
        "lambda_max": 2.0,
        "label": "moderate horizon and weak Y growth",
    },
]

search_records = []
for candidate in candidate_parameters:
    candidate_lambdas = 10.0 ** np.linspace(candidate["lambda_min"], candidate["lambda_max"], J)
    result = simulate_regime(
        candidate["T_value"], J, candidate_lambdas, q_values,
        candidate["p_value"], m, candidate["g_value"], X0, Y0,
        requested_dt, n_store_target_value=1, y_mode="linear", seed=random_seed,
    )
    ok, peak, shares = is_middle_unimodal(result["Y"][-1])
    search_records.append((ok, peak, shares, candidate))

for ok, peak, shares, candidate in search_records:
    print(candidate["label"])
    print(f"  middle-unimodal: {ok}, peak class: {peak + 1}")
    print(f"  final Y shares: {np.round(shares, 3)}")

best_ok, best_peak, best_shares, best_candidate = next(record for record in search_records if record[0])
if HAS_MATPLOTLIB:
    plt.figure(figsize=(7, 4))
    plt.bar(class_index, best_shares)
    plt.axvline(best_peak + 1, color="black", linestyle="--", label="maximum")
    plt.title("Example parameter regime with a middle maximum")
    plt.xlabel("class j")
    plt.ylabel("final Y share")
    plt.legend()
    plt.show()
else:
    render_grouped_bar_svg(class_index, [best_shares], ["Y share"], "Example parameter regime with a middle maximum", "class j", "final Y share", "ode_middle_regime.svg")

## Observation

The baseline `gY_j` model amplifies any class that has already accumulated `Y` mass, so the final `Y` distribution can become strongly skewed. Replacing `gY_j` by `g sqrt(Y_j)` weakens that amplification for large `Y_j`, and the stochastic model adds pathwise fluctuations while the full-truncation update keeps the non-negativity constraint. The bounded parameter search finds a short-horizon regime where the final `Y_j` class histogram has an internal maximum rather than an endpoint maximum.